Montgomery County Maryland Wine Sales Dashboard V2

Module imports

In [1]:
import pandas as pd
import numpy as np
import requests
import PyPDF2
import io
import re
import sqlite3
import time
import datetime
import plotly.graph_objects as go
import plotly.express as px
import warnings
import sys
import pickle
import os
import datetime
import streamlit as st
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from difflib import SequenceMatcher
from datetime import datetime
sys.path.append('./utils')
import fuzzy_supplier_matching as fuzzy
import data_exploration_utils as deu
import wine_classification_utils as wcu
import wine_review_matching_utils as wrmu
import inspect
from difflib import SequenceMatcher
from datetime import datetime
from openpyxl import Workbook
from io import StringIO
warnings.filterwarnings('ignore')

File imports

In [2]:
# GitHub raw URL
base_url = "https://raw.githubusercontent.com/ac604605/Montgomery_County_Dashboard/main/"

print("Loading datasets from GitHub repository...")
print("=" * 60)

try:
    # Load standard datasets
    Distributors_Virginia_Three_Main = pd.read_csv(base_url + "data/Distributors_Virginia_Three_Main.csv")
    wine_producers = pd.read_csv(base_url + "data/wine_producers.csv")
    Warehouse_and_Retail_Sales = pd.read_csv(base_url + "data/Warehouse_and_Retail_Sales.csv")
    Wine_Review_Data = pd.read_csv(base_url + "data/winemag-data-130k-v2.csv/winemag-data-130k-v2.csv")
    
    # Load suppliers with data quality fix
    print("Loading and fixing supplier data structure...")
    correct_columns = ['License_ID', 'Trade Name', 'Address', 'City', 'State', 'Zip_Code', 'Report_Type']
    Suppliers_Fixed = pd.read_csv(
        base_url + "data/Suppliers_Importers_Retailers.csv",
        header=0,
        names=correct_columns,
        usecols=range(7),
        dtype={'License_ID': str, 'Zip_Code': str}
    )
    
    # Professional summaries
    datasets = [
        (Distributors_Virginia_Three_Main, "Virginia Distributors"),
        (wine_producers, "Wine Producers"),
        (Warehouse_and_Retail_Sales, "Sales Transactions"),
        (Wine_Review_Data, "Wine Reviews"),
        (Suppliers_Fixed, "Supplier Directory (Fixed)")  # Note the "Fixed" indicator
    ]
    
    for df, name in datasets:
        print(f"{name:<25} │ {df.shape[0]:>8,} rows × {df.shape[1]:>2} cols │ {df.memory_usage(deep=True).sum() / 1024**2:>6.1f} MB")
    
    # Quick validation for suppliers
    report_types = Suppliers_Fixed['Report_Type'].nunique()
    print(f"Supplier validation: {report_types} unique report types identified")
    
    print("=" * 60)
    print(f"Successfully loaded {len(datasets)} datasets with data quality fixes applied")
    
except Exception as e:
    print(f"Error loading data: {e}")

Loading datasets from GitHub repository...
Loading and fixing supplier data structure...
Virginia Distributors     │   14,279 rows ×  4 cols │    2.4 MB
Wine Producers            │    2,324 rows ×  7 cols │    0.9 MB
Sales Transactions        │  307,645 rows ×  9 cols │   87.5 MB
Wine Reviews              │  129,971 rows × 14 cols │  118.5 MB
Supplier Directory (Fixed) │   78,067 rows ×  7 cols │   31.2 MB
Supplier validation: 19 unique report types identified
Successfully loaded 5 datasets with data quality fixes applied


Now with everything loaded, I will begin data cleaning and refinement to suit the needs of this specific dashboard. 

After investigating the sales data, there are a few cleaning steps that need to take place. First will be removing all values that are not wine and beer items carried by distributors. Second will be ensuring item codes are numeric for easier processesing. Finally, some idividual values will be changed and anything that isn't wine or beer will be removed. I will also be removing the keg versions of wines and beer, as those would introduce greater scope that I do not wish to manage for a simple portfolio. 

In [3]:
# Create working copy for processing
print("Creating working copy of sales data...")
df_working = Warehouse_and_Retail_Sales.copy()
print(f"Working dataset: {df_working.shape[0]:,} rows × {df_working.shape[1]} columns")

# Run your enhanced data cleaning utilities
print("\nStarting data cleaning pipeline...")
df_clean, cleaning_report = deu.run_complete_item_code_standardization(
    df_working, 
    item_types_to_keep=['WINE', 'BEER']
)

# Show cleaning results
print(f"\nCleaning Results:")
print(f"   Original: {cleaning_report['original_shape']}")
print(f"   Cleaned:  {cleaning_report['final_shape']}")
print(f"   Retention: {cleaning_report['summary']['data_retention_pct']:.1f}%")

Creating working copy of sales data...
Working dataset: 307,645 rows × 9 columns

Starting data cleaning pipeline...
 COMPLETE ITEM CODE STANDARDIZATION PIPELINE
 Processing dataset with 307,645 rows and 9 columns
🧹 STEP 1: REMOVING MISSING SUPPLIER DATA
 Original dataset shape: (307645, 9)
 Rows with missing SUPPLIER: 167
 Missing SUPPLIER percentage: 0.05%

 Cleaning complete!
 New dataset shape: (307478, 9)
  Rows removed: 167
✓ Null SUPPLIER remaining: 0
 Data reduction: 0.05%

 STEP 2: ANALYZING NON-NUMERIC ITEM CODE PATTERNS
 Total items analyzed: 307,478
🔤 Non-numeric item codes found: 7
 Non-numeric percentage: 0.00%

 Analyzing patterns in non-numeric codes...
 Pattern analysis:
   Most common format: [numbers][letters]
   Suffixes found:
     'A': 7 occurrences

 Sample non-numeric codes (showing up to 10):
   97024A - GLUTENBERG PALE ALE - 16OZ CAN...
   50029A - AVERY ELLIES BROWN 4/6 12OZ CAN...
   300662A - DR STONERS SMOKY HERB WHISKEY 750ML...
   348152A - VIRGINIA BLAC

Step 2: Some item codes have non-numeric values. After investigation, I made a decision to consolidate items with and without non-numberic codes into the same item number. This will help simplify the final graph without adding undue work and with minimal lost data granularity. 

In [4]:
# First, recreate the non_numeric variable
non_numeric = df_clean_Warehouse_and_Retail_Sales[~df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.isdigit()]
print(f"Non-numeric item codes found: {len(non_numeric)}")

# Now you can run your pattern analysis
print("Patterns in non-numeric item codes:")
print(non_numeric['ITEM CODE'].str.extract(r'(\d+)([A-Za-z]+)'))

# Count different suffixes
suffixes = non_numeric['ITEM CODE'].str.extract(r'\d+([A-Za-z]+)')
print("\nSuffixes found:")
print(suffixes[0].value_counts())

NameError: name 'df_clean_Warehouse_and_Retail_Sales' is not defined

In [ ]:
# Check if there are numeric versions of these items
print("Checking for numeric versions of 'A' suffix items...")

for item_code in non_numeric['ITEM CODE'].head(10):
    if item_code.endswith('A'):
        base_code = item_code[:-1]  # Remove the 'A'
        
        # Check if the base code exists
        base_exists = df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.contains(f'^{base_code}$').any()
        
        print(f"{item_code} (base: {base_code}) - Base exists: {base_exists}")
        
        if base_exists:
            base_item = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == base_code]
            a_item = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == item_code]
            
            print(f"  Base: {base_item['ITEM DESCRIPTION'].iloc[0]}")
            print(f"  A version: {a_item['ITEM DESCRIPTION'].iloc[0]}")
            print("---")

In [ ]:
# Step 1: Identify all A-suffix items that have base versions
a_suffix_items = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.endswith('A')].copy()
a_suffix_items['BASE_CODE'] = a_suffix_items['ITEM CODE'].str[:-1]  # Remove the 'A'

print(f"Found {len(a_suffix_items)} items with 'A' suffix")

# Step 2: Check which ones have corresponding base versions
base_codes_exist = a_suffix_items['BASE_CODE'].isin(df_clean_Warehouse_and_Retail_Sales['ITEM CODE'])
consolidatable = a_suffix_items[base_codes_exist].copy()

print(f"Can consolidate {len(consolidatable)} items (have matching base codes)")

# Step 3: For each A-suffix item, copy the base version's key columns
for idx, row in consolidatable.iterrows():
    base_code = row['BASE_CODE']
    a_code = row['ITEM CODE']
    
    # Find the base version
    base_row = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == base_code].iloc[0]
    
    # Update the A-version to match base version for key columns
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'SUPPLIER'] = base_row['SUPPLIER']
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'ITEM CODE'] = base_row['ITEM CODE'] 
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'ITEM DESCRIPTION'] = base_row['ITEM DESCRIPTION']
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'ITEM TYPE'] = base_row['ITEM TYPE']

print("Consolidation complete!")

# Step 4: Verify the changes
print("\nVerification - should now see duplicates:")
sample_base = consolidatable['BASE_CODE'].iloc[0]
consolidated_items = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == sample_base]
print(f"Items with code {sample_base}: {len(consolidated_items)}")
print(consolidated_items[['ITEM CODE', 'ITEM DESCRIPTION']].head())

In [ ]:
# Find ALL non-numeric item codes in your clean dataset
print("Finding all non-numeric item codes...")

# Check which ones are NOT purely numeric
non_numeric = df_clean_Warehouse_and_Retail_Sales[~df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.isdigit()]
print(f"Non-numeric item codes found: {len(non_numeric)}")

if len(non_numeric) > 0:
    print("\nUnique non-numeric item codes:")
    print(non_numeric['ITEM CODE'].value_counts())
    
    print("\nSample of these items:")
    print(non_numeric[['ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE']].head(10))

In [ ]:
# Convert to numeric
df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] = pd.to_numeric(df_clean_Warehouse_and_Retail_Sales['ITEM CODE'])

# Verify the conversion
print(f"\nAfter conversion:")
print(f"New ITEM CODE dtype: {df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].dtype}")
print(f"Sample item codes: {df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].head().tolist()}")
print(f"Any NaN values created: {df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].isnull().sum()}")

Step 3: After investigating the various item types available in the report, I have decided to remove all except wine and beer. This will lose some sales from wine and beer kegs, but it will simplify the report. Again this is being done for simplicity. 

In [ ]:
# Check current counts
print("Before filtering:")
print(df_clean_Warehouse_and_Retail_Sales['ITEM TYPE'].value_counts())
print(f"Total rows: {len(df_clean_Warehouse_and_Retail_Sales)}")

# Filter to keep only WINE and BEER
df_clean_Warehouse_and_Retail_Sales = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM TYPE'].isin(['WINE', 'BEER'])]

# Check results
print("\nAfter filtering:")
print(df_clean_Warehouse_and_Retail_Sales['ITEM TYPE'].value_counts())
print(f"Total rows: {len(df_clean_Warehouse_and_Retail_Sales)}")

# Calculate what was removed
wine_beer_total = 187640 + 42413
print(f"\nRows kept: {len(df_clean_Warehouse_and_Retail_Sales)} (WINE + BEER)")
print(f"Rows removed: {307478 - len(df_clean_Warehouse_and_Retail_Sales)} (LIQUOR, KEGS, etc.)")

I will also perform some joins and various cleaning of supporting data tables meant to make brand ownership rights more clear. 

In [ ]:
df_clean_Warehouse_and_Retail_Sales_enhanced = run_supplier_enrichment(df_clean_Warehouse_and_Retail_Sales, 
                                     Suppliers_Fixed, test_mode=False)

In [ ]:
# Check which suppliers got matched
matched_suppliers = df_clean_Warehouse_and_Retail_Sales_enhanced[
    (df_clean_Warehouse_and_Retail_Sales_enhanced['SUPPLIER_MATCH_SCORE'] >= 0.8) & 
    (df_clean_Warehouse_and_Retail_Sales_enhanced['SUPPLIER_REPORT_TYPE'] == 'Wholesale Wine Distributors')
]

print("Your Wholesale Wine Distributors:")
print(matched_suppliers['SUPPLIER'].value_counts())

print("\nSample matched data:")
print(matched_suppliers[['SUPPLIER', 'MATCHED_SUPPLIER_NAME', 'SUPPLIER_MATCH_SCORE']].head(10))

Now I will begin manual exploration of both the Wine Review Data. 

Uncomment various lines to review data. The below functions from the accompanying data utilities module are available for further data set analysis. 

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed

#uncomment below lines to change report
#df.head()
#df.info()
#df.describe()
#df.isnull().sum()
#df['country'].value_counts()
#deu.investigate_missing_values_manually(df)

After reviewing the wine review data, it appears this will serve as an adequate database to gather missing country data for our sales table. For the sake of simplicity, I will combine all columns from this table to matching wines in our sales table. Columns we do not need can be filtered out later. 

Step 1: Testing sample search terms on the wine review data. 

In [ ]:
search_terms = ["SANTA MARGHERITA", "ROBERT MONDAVI", "CUPCAKE", "CAKEBREAD", "BUTTER"]

for term in search_terms:
    print(f"\n--- Searching for '{term}' ---")
    matches = Wine_Review_Data[Wine_Review_Data['title'].str.contains(term, case=False, na=False)]
    print(f"Found {len(matches)} matches")
    if len(matches) > 0:
        print("Sample matches:")
        for idx, row in matches.head(3).iterrows():
            print(f"  - {row['title']} | Country: {row['country']}")

After testing simple search results above, I will begin to create a search algorithm below. First up, testing a simple scoring function to differentiate between wines with similar names. 

In [ ]:
def score_match(search_term, wine_title):
    """Score how well a search term matches a wine title."""
    
    # Convert to uppercase for case-insensitive comparison
    search_term = search_term.upper()
    wine_title = wine_title.upper()
    
    score = 0
    # Check if it's an exact producer match
    if wine_title.startswith(search_term):
        score += 100
    
    return score

def find_best_matches(search_term, Wine_Review_Data, top_n=5):
    """
    Find the best matching wines for a search term and return them with scores
    """
    matches = []
    
    # First, find all wines that contain our search term
    potential_matches = Wine_Review_Data[Wine_Review_Data['title'].str.contains(search_term, case=False, na=False, regex=False)]
    
    # Now score each potential match
    for index, row in potential_matches.iterrows():
        score = score_match(search_term, row['title'])
        matches.append({
            'title': row['title'],
            'country': row['country'],
            'score': score
        })
    
    # Sort by score (highest first) and return top matches
    matches = sorted(matches, key=lambda x: x['score'], reverse=True)
    return matches[:top_n]

# Test the functions work together
print("Testing complete matching system:")
print("=" * 40)

test_terms = ["SANTA MARGHERITA", "ROBERT MONDAVI", "CUPCAKE"]
for term in test_terms:
    results = find_best_matches(term, Wine_Review_Data, top_n=3)
    print(f"\nTop matches for {term}:")
    for match in results:
        print(f"  Score: {match['score']} | {match['title']} | Country: {match['country']}")

In [ ]:
def add_all_match_columns_with_backup(df_clean_Warehouse_and_Retail_Sales, Wine_Review_Data, resume_from_checkpoint=None):
    """
    Enhanced version with better backup and resume capability
    """
    import time
    import datetime
    import pickle
    import os
    
    print("Starting country/enrichment matching with enhanced backup...")
    
    # Resume from checkpoint if provided
    if resume_from_checkpoint and os.path.exists(resume_from_checkpoint):
        print(f"🔄 Resuming from checkpoint: {resume_from_checkpoint}")
        df_with_matches = pd.read_pickle(resume_from_checkpoint)
        
        # Find where to resume
        wine_mask = df_with_matches['ITEM TYPE'] == 'WINE'
        processed_wines = df_with_matches[wine_mask & (df_with_matches['PRODUCER_FOUND'] != '')]
        start_idx = len(processed_wines)
        print(f"📍 Resuming from wine #{start_idx:,}")
    else:
        df_with_matches = df_clean_Warehouse_and_Retail_Sales.copy()
        start_idx = 0
        
        # Initialize columns (only if starting fresh)
        Wine_Review_Data_columns = Wine_Review_Data.columns.tolist()
        print(f"Will add {len(Wine_Review_Data_columns)} columns from Wine_Review_Data dataset")
        
        for col in Wine_Review_Data_columns:
            if col in df_with_matches.columns:
                new_col_name = f"Wine_Review_Data_{col}"
            else:
                new_col_name = col
            
            if col.lower() in ['country']:
                df_with_matches[new_col_name] = 'Unknown'
            elif col.lower() in ['points', 'price', 'score', 'rating']:
                df_with_matches[new_col_name] = 0
            else:
                df_with_matches[new_col_name] = ''
        
        # Initialize tracking columns
        df_with_matches['PRODUCER_FOUND'] = ''
        df_with_matches['MATCH_CONFIDENCE'] = 0
        df_with_matches['TOTAL_MATCHES'] = 0
    
    description_cache = {}
    matched_count = len(df_with_matches[df_with_matches['PRODUCER_FOUND'] != ''])
    cache_hits = 0
    
    wine_mask = df_with_matches['ITEM TYPE'] == 'WINE'
    wine_items = df_with_matches[wine_mask].iloc[start_idx:]  # Resume from checkpoint
    total_wines = len(df_with_matches[wine_mask])
    
    print(f"Processing {len(wine_items)} remaining wines (of {total_wines} total)...")
    
    start_time = time.time()
    
    for i, (idx, row) in enumerate(wine_items.iterrows()):
        current_position = start_idx + i
        desc = row['ITEM DESCRIPTION']
        
        # Skip if already processed
        if row['PRODUCER_FOUND'] != '':
            continue
        
        # Check cache first
        if desc in description_cache:
            result = description_cache[desc]
            cache_hits += 1
        else:
            # Extract search term from description (adjust logic as needed)
            search_term = ' '.join(desc.split()[:2])  # First 2 words
            matches = find_best_matches(search_term, Wine_Review_Data, top_n=1)
    
            if matches and len(matches) > 0:  # This line needs to be indented
                # Get the full row from Wine_Review_Data for the best match
                best_match = matches[0]
                matched_rows = Wine_Review_Data[Wine_Review_Data['title'] == best_match['title']]
                
                if len(matched_rows) > 0:
                    matched_row_dict = matched_rows.iloc[0].to_dict()
                    result = {
                        'producer_found': search_term,
                        'confidence': best_match['score'],
                        'total_matches': len(matches),
                        'matched_row': matched_row_dict
                    }
                else:
                    result = None
            else:
                result = None
    
            description_cache[desc] = result
            
        if result:  # This line should align with the 'if desc in description_cache:' above
            # Add Wine_Review_Data columns
            if 'matched_row' in result and result['matched_row'] is not None:
                matched_row = result['matched_row']
                Wine_Review_Data_columns = Wine_Review_Data.columns.tolist()
                
                for col in Wine_Review_Data_columns:
                    if col in df_clean_Warehouse_and_Retail_Sales.columns:
                        new_col_name = f"Wine_Review_Data_{col}"
                    else:
                        new_col_name = col
                    
                    value = matched_row.get(col, '')
                    df_with_matches.at[idx, new_col_name] = value
            
            # Update tracking columns
            df_with_matches.at[idx, 'PRODUCER_FOUND'] = result.get('producer_found', '')
            df_with_matches.at[idx, 'MATCH_CONFIDENCE'] = result.get('confidence', 0)
            df_with_matches.at[idx, 'TOTAL_MATCHES'] = result.get('total_matches', 0)
            matched_count += 1
        
        # Save checkpoint every 500 items (more frequent)
        if (current_position + 1) % 500 == 0:
            checkpoint_filename = f'wine_matching_checkpoint_{current_position+1}.pkl'
            df_with_matches.to_pickle(checkpoint_filename)
            print(f"*** 💾 CHECKPOINT SAVED: {checkpoint_filename} ***")
        
        # Progress reporting every 100 items
        if (current_position + 1) % 100 == 0:
            elapsed_time = time.time() - start_time
            processed = current_position + 1
            
            if processed > start_idx:
                avg_time = elapsed_time / (processed - start_idx)
                remaining = total_wines - processed
                eta_seconds = remaining * avg_time
                eta = str(datetime.timedelta(seconds=int(eta_seconds)))
            else:
                eta = "Calculating..."
            
            print(f"Progress: {processed:,}/{total_wines:,} ({processed/total_wines*100:.1f}%)")
            print(f"  Matches: {matched_count:,} | ETA: {eta}")
    
    # Final save
    final_filename = f'wine_matching_FINAL_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}.pkl'
    df_with_matches.to_pickle(final_filename)
    print(f"🎉 FINAL RESULTS SAVED: {final_filename}")
    
    return df_with_matches

After investigating the resulting dataframe, there seem to be some matching inefficiencies. I'm going to work on identifying and improving those below. First, I'll review all table data and check that column names and table names are matched correctly. There were some discrepancies between naming conventions from various sites. 

full_results, sales_map, review_map = wrmu.run_wine_review_matching(
    df_clean_Warehouse_and_Retail_Sales, Wine_Review_Data, threshold=0.6, test_mode=False
)

In [ ]:
# Save as pickle (recommended for data analysis)
#full_results.to_pickle('wine_sales_with_reviews_FINAL.pkl')

# To load later:
df = pd.read_pickle('wine_sales_with_reviews_FINAL.pkl')

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
#df=df_loaded
#df=df_final
#df=variety_counts
df=df

#uncomment below lines to change report
#df.head(25)
#df.info()
#df.describe()
#df[df['review_country'] == ''].head()
df.head()
#pd.set_option('display.max_rows', None)
#df['ITEM DESCRIPTION'].value_counts()
#deu.investigate_missing_values_manually(df)

In [ ]:
# Get the unknowns from enhanced classification
enhanced_results = missing_varieties['ITEM DESCRIPTION'].apply(classify_missing_variety_enhanced)
still_unknown = missing_varieties[enhanced_results == 'Unknown']

# Get additional context columns
unknown_with_context = still_unknown[['ITEM DESCRIPTION', 'SUPPLIER', 'review_variety', 
                                     'review_variety_consolidated', 'extracted_variety', 
                                     'review_country', 'review_winery', 'WINE_NAME_EXTRACTED']].copy()

# Sort by item description frequency
unknown_counts = still_unknown['ITEM DESCRIPTION'].value_counts()

print("Top unknown item descriptions with additional context:")
print("=" * 100)

for i, (description, count) in enumerate(unknown_counts.head(30).items(), 1):
    print(f"\n{i:2d}. ({count:2d}x) {description}")
    
    # Get a sample record for this description to show context
    sample = unknown_with_context[unknown_with_context['ITEM DESCRIPTION'] == description].iloc[0]
    
    # Only show non-empty additional info
    context_info = []
    if pd.notna(sample['SUPPLIER']) and sample['SUPPLIER'].strip():
        context_info.append(f"Supplier: {sample['SUPPLIER']}")
    if pd.notna(sample['review_variety']) and sample['review_variety'].strip():
        context_info.append(f"Review Variety: {sample['review_variety']}")
    if pd.notna(sample['review_country']) and sample['review_country'].strip():
        context_info.append(f"Country: {sample['review_country']}")
    if pd.notna(sample['review_winery']) and sample['review_winery'].strip():
        context_info.append(f"Winery: {sample['review_winery']}")
    if pd.notna(sample['WINE_NAME_EXTRACTED']) and sample['WINE_NAME_EXTRACTED'].strip():
        context_info.append(f"Wine Name: {sample['WINE_NAME_EXTRACTED']}")
    
    if context_info:
        print(f"    Context: {' | '.join(context_info)}")
    else:
        print(f"    Context: No additional info available")

print(f"\nTotal unique unknown descriptions: {len(unknown_counts)}")
print(f"Total unknown records: {unknown_counts.sum()}")

In [ ]:
# Check existing variety categories in your full dataset
print("Current variety categories in your database:")
all_varieties = df['final_variety'].value_counts()
print(f"Total unique varieties: {len(all_varieties)}")
print("\nTop 50 varieties:")
print(all_varieties.head(50))

# Check for specific patterns that might help with edge cases
print("\n" + "="*50)
print("Checking for specific patterns in existing data:")

# Check for fruit wine patterns
fruit_patterns = all_varieties[all_varieties.index.str.contains('Fruit|Berry|Peach|Pomegranate', case=False, na=False)]
if len(fruit_patterns) > 0:
    print(f"\nFruit wine patterns found:")
    print(fruit_patterns)
else:
    print("\nNo fruit wine patterns found in existing varieties")

# Check for Italian blend patterns
italian_patterns = all_varieties[all_varieties.index.str.contains('Italian|Tuscany|Chianti', case=False, na=False)]
if len(italian_patterns) > 0:
    print(f"\nItalian wine patterns found:")
    print(italian_patterns)
else:
    print("\nNo specific Italian patterns found")

# Check for moscato patterns
moscato_patterns = all_varieties[all_varieties.index.str.contains('Moscato|Muscat', case=False, na=False)]
if len(moscato_patterns) > 0:
    print(f"\nMoscato patterns found:")
    print(moscato_patterns)
else:
    print("\nNo Moscato variations found")

In [ ]:
# Step 1: Create the final classification function (copy the big function from earlier)
def classify_missing_variety_final(item_description):
    """
    Final enhanced variety extraction from item descriptions
    """
    if not item_description:
        return 'Unknown'
    
    desc = item_description.upper()
    
    # Direct variety matches
    if 'GRIGIO' in desc or 'P/GRIGIO' in desc:
        return 'Pinot Grigio'
    elif 'CHARDONNAY' in desc or 'CHARD' in desc:
        return 'Chardonnay'
    elif 'CABERNET' in desc:
        if 'SAUVIGNON' in desc:
            return 'Cabernet Sauvignon'
        else:
            return 'Cabernet Sauvignon'
    elif 'MERLOT' in desc:
        return 'Merlot'
    elif 'SAUVIGNON' in desc and 'CABERNET' not in desc:
        return 'Sauvignon Blanc'
    elif 'PINOT' in desc and 'NOIR' in desc:
        return 'Pinot Noir'
    elif 'P/NERO' in desc:  # Italian Pinot Nero
        return 'Pinot Noir'
    elif 'PINOT' in desc:
        return 'Pinot Grigio'
    elif 'SYRAH' in desc:
        return 'Syrah'
    elif 'RIESLING' in desc or 'RIE' in desc:  # Added RIE for German abbreviations
        return 'Riesling'
    elif 'MOSCATO' in desc:
        return 'Moscato'
    elif 'TEMPRANILLO' in desc:
        return 'Tempranillo'
    
    # More variety patterns
    elif 'MALB' in desc:
        return 'Malbec'
    elif 'TORRONTES' in desc:
        return 'Torrontés'
    elif 'GRUNER VELT' in desc or 'GRÜNER VELT' in desc:
        return 'Grüner Veltliner'
    elif 'ALBR' in desc or 'ALBARIÑO' in desc:
        return 'Albariño'
    elif 'PRIMITIVO' in desc:
        return 'Primitivo'
    elif 'SAPERAVI' in desc:
        return 'Saperavi'
    elif 'VIDAL' in desc:
        return 'Vidal Blanc'
    elif 'PICPOUL' in desc:
        return 'Picpoul'
    elif 'NEGRA' in desc:
        return 'Red Blend'
    
    # Regional/style indicators
    elif 'POUILLY FUME' in desc:
        return 'Sauvignon Blanc'
    elif 'MONTEPULCIANO' in desc or 'MONTEPUL' in desc or 'MONT/PUL' in desc:
        return 'Montepulciano'
    elif 'CARMENERE' in desc:
        return 'Carmenère'
    elif 'VERMENTINO' in desc:
        return 'Vermentino'
    elif 'ST EMILION' in desc:
        return 'Red Blend'
    elif 'SUPER TUSCAN' in desc:
        return 'Red Blend'
    elif 'PROVENCE' in desc and ('RSE' in desc or 'ROSE' in desc):
        return 'Rosé'
    elif 'PECORION' in desc:
        return 'Pecorino'
    elif 'ICE' in desc and 'VIDAL' in desc:
        return 'Ice Wine'
    elif 'VALPOLICELLA' in desc or 'RIPASSA VAL' in desc:
        return 'Valpolicella'
    elif 'GSM' in desc:  # Grenache/Syrah/Mourvèdre
        return 'Red Blend'
    
    # NEW: Specific wine name patterns from your list
    elif 'JAZZ BERRY' in desc:
        return 'Fruit Wine'
    elif 'APOTHIC' in desc and 'DARK' in desc:
        return 'Red Blend'
    elif 'CARLO ROSSI PAISANO' in desc:
        return 'Red Blend'
    elif 'SWEET WALTER WHT' in desc:
        return 'White Blend'
    elif 'CHOCOVINE' in desc:
        return 'Red Blend'
    elif 'GOLLY WOBBLER' in desc:
        return 'Red Blend'
    elif 'PECHE IMPERIALE' in desc:
        return 'Moscato'
    elif 'ORIN SWIFT' in desc:
        return 'Red Blend'
    elif 'ANCIANO' in desc and 'RES' in desc:
        return 'Tempranillo'
    elif 'POMEGRANTE' in desc or 'POMEGRANATE' in desc:
        return 'Fruit Wine'
    elif 'PALAZZO DEL TORRE' in desc:
        return 'Red Blend'
    elif 'RES DUCALE' in desc:  # Ruffino Chianti
        return 'Chianti'
    elif 'SEY/CH/VID' in desc:  # Seyval/Chardonnay/Vidal blend
        return 'White Blend'
    
    # Sparkling wines
    elif any(x in desc for x in ['SPARK', 'SPUMANTE', 'ASTI', 'CHAMPAGNE', 
                                 'BL DE NOIR', 'BLANC DE NOIRS', 'CORDON NEGRO',
                                 'ACE OF SPADE']):
        return 'Sparkling Blend'
    elif 'PROSECCO' in desc:
        return 'Prosecco'
    elif 'FREIXENET' in desc:
        return 'Sparkling Blend'
    
    # Fortified wines
    elif any(x in desc for x in ['VERMOUTH', 'VERM', 'SHERRY', 'PORT', 'MADEIRA',
                                 'MARSALA', 'RAINWATER', 'BRISTOL CREAM']):
        return 'Vermouth' if 'VERM' in desc else 'Sherry' if any(x in desc for x in ['SHERRY', 'BRISTOL CREAM']) else 'Madeira' if any(x in desc for x in ['MADEIRA', 'RAINWATER']) else 'Marsala' if 'MARSALA' in desc else 'Port'
    
    # Special categories
    elif 'MEAD' in desc:
        return 'Mead'
    elif 'SAKE' in desc or any(x in desc for x in ['TOZAI', 'KOOK SOON']):
        return 'Sake'
    elif any(x in desc for x in ['MAKKOLI', 'MAKGEOLLI']):
        return 'Sake'  # Korean rice wine, but closest category
    elif 'MD 20/20' in desc:
        return 'Fortified Wine'
    elif 'COLD DUCK' in desc:
        return 'Sparkling Blend'
    elif any(x in desc for x in ['PLUM', 'CHOYA']):
        return 'Fruit Wine'
    
    # Color-based fallbacks
    elif any(word in desc for word in ['RED', 'ROUGE', 'ROSSO', 'TINTO', 'RGE']):
        return 'Red Blend'
    elif any(word in desc for word in ['WHITE', 'BLANC', 'BLANCO', 'BIANCO']):
        return 'White Blend'
    elif any(word in desc for word in ['ROSE', 'RSE', 'ROSÉ']):
        return 'Rosé'

# Quick additions to the classification function:

    # Easy sparkling wins
    elif 'LUC BELAIRE' in desc or 'BELAIRE' in desc:
        return 'Sparkling Blend'
    elif 'VILLA JOLANDA' in desc:
        return 'Sparkling Blend'  # Italian sparkling producer

    # More abbreviation patterns  
    elif 'P/TAGE' in desc:  # Pinotage
        return 'Pinotage'
    elif 'GRUN VELT' in desc or 'GRÜN VELT' in desc:  # Grüner Veltliner
        return 'Grüner Veltliner'
    elif 'GARN' in desc:  # Garnacha
        return 'Garnacha'

    # Rice wine patterns (Korean/Asian)
    elif any(x in desc for x in ['BAEK', 'SOO', 'BOK', 'WHA']):
        return 'Sake'  # Your closest category for rice wines
    
    # More wine name patterns
    elif 'SANT GRIA' in desc or "SANT' GRIA" in desc:
        return 'Sangria'  # Yago makes sangria
    elif 'HONEY WINE' in desc:
        return 'Mead'  # Enat honey wine
    elif 'BERTANI VAL' in desc:
        return 'Valpolicella'  # Bertani Valpolicella
    elif 'WILDLY WICKED' in desc:
        return 'Red Blend'  # Live a Little brand
    
    else:
        return 'Unknown'
    pass

# Step 2: Apply the function to missing varieties only
missing_mask = df['final_variety'] == ''
df.loc[missing_mask, 'final_variety'] = df.loc[missing_mask, 'ITEM DESCRIPTION'].apply(classify_missing_variety_final)

# Step 3: Verify the results
print("Updated variety counts:")
print(df['final_variety'].value_counts().head(20))

print(f"\nBefore: {missing_mask.sum()} missing varieties")
print(f"After: {(df['final_variety'] == 'Unknown').sum()} unknown varieties") 
print(f"Still empty: {(df['final_variety'] == '').sum()} empty varieties")

# Step 4: Optional - convert 'Unknown' to empty string if you prefer
# df.loc[df['final_variety'] == 'Unknown', 'final_variety'] = ''

In [ ]:
# See what variety-related columns exist
variety_cols = [col for col in df.columns if 'variety' in col.lower()]
print("Available variety columns:", variety_cols)

In [ ]:
def classify_wine_color(variety):
    """
    Classify wine variety into color categories
    """
    if not variety or variety.strip() == '':
        return 'Unknown'
    
    variety = variety.strip().lower()
    
    # Red wines
    red_varieties = {
        'red blend', 'cabernet sauvignon', 'pinot noir', 'merlot', 'malbec', 
        'syrah', 'zinfandel', 'tempranillo', 'sangiovese', 'nebbiolo',
        'cabernet franc', 'montepulciano', 'chianti', 'gamay', 'barbera',
        'garnacha', 'portuguese red', 'shiraz', 'nero d\'avola', 'petite sirah',
        'monastrell', 'amarone', 'primitivo', 'pinotage', 'corvina, rondinella, molinara',
        'saperavi', 'nerello mascalese', 'carmenère', 'grenache', 'syrah-viognier',
        'bonarda', 'dolcetto', 'mencía', 'tannat-cabernet', 'cannonau',
        'valpolicella', 'agiorgitiko', 'carignan', 'sagrantino', 'tannat',
        'negroamaro', 'frappato', 'xinomavro', 'malbec-merlot', 'valdiguié',
        'blaufränkisch', 'malvasia nera', 'touriga nacional', 'gamza',
        'cabernet sauvignon-merlot', 'tempranillo-merlot', 'petit verdot',
        'st. laurent', 'teroldego', 'sousão', 'graciano', 'lagrein',
        'chambourcin', 'plavac mali', 'portuguiser', 'monica', 'bobal',
        'mavrud', 'corvina', 'syrah-cabernet', 'feteasca neagra', 'refosco',
        'lemberger', 'pinot noir-gamay', 'syrah-cabernet sauvignon', 
        'dornfelder', 'garnacha tintorera', 'cabernet franc-cabernet sauvignon',
        'piedirosso', 'vranec', 'kekfrankos', 'gaglioppo', 'alicante bouschet',
        'duras', 'schiava', 'argaman', 'prieto picudo', 'papaskarasi',
        'roviello', 'cinsault', 'mandilaria', 'melnik', 'monastrell-syrah',
        'susumaniello'
    }
    
    # White wines
    white_varieties = {
        'chardonnay', 'sauvignon blanc', 'pinot grigio', 'white blend',
        'moscato', 'riesling', 'albariño', 'viognier', 'portuguese white',
        'gewürztraminer', 'chenin blanc', 'garganega', 'verdejo', 'catarratto',
        'viura', 'vinho verde', 'grillo', 'grüner veltliner', 'cortese',
        'torrontés', 'carricante', 'melon', 'moschofilero', 'picpoul',
        'ugni blanc-colombard', 'greco', 'falanghina', 'rkatsiteli',
        'vermentino', 'friulano', 'muscadet', 'assyrtiko', 'sémillon',
        'malvasia', 'savatiano', 'godello', 'verdicchio', 'verdejo-viura',
        'german white blend', 'inzolia', 'vernaccia', 'torbato', 'arneis',
        'rieslaner', 'furmint', 'avesso', 'kisi', 'silvaner', 'pecorino',
        'petit manseng', 'ribolla gialla', 'müller-thurgau', 'grechetto',
        'symphony', 'robola', 'viognier-chardonnay', 'trebbiano', 'turbiana',
        'zibibbo', 'traminette', 'roussanne', 'auxerrois', 'erbaluce',
        'passerina', 'soave', 'kerner', 'vidal blanc', 'greco bianco',
        'antão vaz', 'malvasia istriana', 'chinuri', 'žilavka', 'mtsvane',
        'scheurebe', 'malagousia', 'chenin blanc-chardonnay', 'albana',
        'loureiro', 'jacquère', 'vilana', 'marsanne-viognier', 'picapoll',
        'garnacha blanca', 'malvasia bianca', 'chardonnay-sauvignon',
        'seyval blanc', 'emir', 'tocai'
    }
    
    # Sparkling (can be red, white, or rosé but handled separately)
    sparkling_varieties = {
        'sparkling blend', 'champagne blend', 'prosecco', 'glera',
        'portuguese sparkling', 'lambrusco', 'lambrusco di sorbara',
        'lambrusco grasparossa', 'brachetto'
    }
    
    # Rosé (including white zinfandel)
    rose_varieties = {
        'rosé', 'white zinfandel'
    }
    
    # Fortified/Dessert wines
    fortified_varieties = {
        'port', 'sherry', 'vermouth', 'madeira', 'ice wine', 'mavrodaphne',
        'malaga', 'dessert wine', 'tokaji', 'white port', 'pedro ximénez',
        'terrantez', 'bual', 'palomino', 'fortified wine'
    }
    
    # Other/Special categories
    other_varieties = {
        'sake', 'fruit wine', 'sangria', 'concord'
    }
    
    # Classification logic
    if variety in red_varieties:
        return 'Red'
    elif variety in white_varieties:
        return 'White'
    elif variety in sparkling_varieties:
        return 'Sparkling'
    elif variety in rose_varieties:
        return 'Rosé'
    elif variety in fortified_varieties:
        return 'Fortified'
    elif variety in other_varieties:
        return 'Other'
    else:
        return 'Unknown'
    
# Apply it
df['wine_color'] = df['final_variety'].apply(classify_wine_color)

print("Wine color distribution:")
print(df['wine_color'].value_counts())

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
#df=df_loaded
#df=df_final
#df=variety_counts

#uncomment below lines to change report
#df.head(25)
df.info()
#df.describe()
#df[df['final_variety'] == ''].head()
#pd.set_option('display.max_rows', None)
#df['ITEM DESCRIPTION'].value_counts()
#deu.investigate_missing_values_manually(df)

In [ ]:
# Save as pickle (recommended for data analysis)
df.to_pickle('wine_data_fully_classified.pkl')

# To load later:
#df = pd.read_pickle('wine_data_fully_classified.pkl')